# Baseline 2 — Vanilla RAG
## GraphRAG Benchmark · Bloomberg Financial News

**Role in benchmark:** Standard vector retrieval — chunk the corpus, embed, retrieve top-k, answer with context.  
This isolates the effect of **retrieval structure**: flat chunk similarity (Vanilla RAG) vs relational subgraph (GraphRAG).

**Fairness guarantees:**
- Same LLM: `mistral-nemo` via Ollama
- Same 5 000-article corpus
- Same 20 questions from `qa_set.json` (generated in Notebook 1)
- Same RAG Triad evaluation via `llama3.2:3b` judge

**Memory strategy:**
- Embedding model: `all-MiniLM-L6-v2` — 22 MB, runs on CPU
- Vector store: pure numpy dot product — no FAISS, no extra install
- Articles are streamed and embedded in batches of 128 to cap RAM
- Kernel RAM stays under ~2 GB even for 5 000 articles

## 1. Setup

The only addition over Notebook 1 is `sentence-transformers`. No torch GPU, no FinBERT.

In [ ]:
!pip install requests datasets tqdm sentence-transformers -q
print("Done.")

## 2. Configuration

**Why these chunking parameters:**
- `CHUNK_SIZE = 512` chars ≈ 2-3 sentences — small enough to be semantically focused, large enough to contain a full entity-relation triple
- `CHUNK_OVERLAP = 64` chars — prevents relation spans being cut at chunk boundaries
- `TOP_K = 5` — same as your platform's RAG pipeline

In [ ]:
import json, os, re, time, requests
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

CFG = {
    "ollama_url"   : "http://localhost:11434",
    "llm_model"    : "mistral-nemo",
    "judge_model"  : "llama3.2:3b",
    "embed_model"  : "sentence-transformers/all-MiniLM-L6-v2",
    "n_articles"   : 5000,
    "chunk_size"   : 512,
    "chunk_overlap": 64,
    "top_k"        : 5,
    "embed_batch"  : 128,   # articles embedded per batch — keep low to cap RAM
    "article_max_chars": 1200,
    "qa_set_path"  : "qa_set.json",
    "results_path" : "baseline_rag_results.json",
    "temperature"  : 0.0,
}

def ollama(model, prompt, timeout=90):
    try:
        r = requests.post(
            f"{CFG['ollama_url']}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": CFG["temperature"]}},
            timeout=timeout,
        )
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        return f"[ERROR: {e}]"

try:
    r = requests.get(f"{CFG['ollama_url']}/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama up. Models: {models}")
except Exception as e:
    print(f"WARNING: Ollama not reachable — {e}")

## 3. Load Shared QA Set

Load the 20 questions generated in Notebook 1.  
**Do not regenerate** — Notebook 2 must answer the exact same questions as Notebook 1 and the GraphRAG pipeline.

In [ ]:
if not os.path.exists(CFG["qa_set_path"]):
    raise FileNotFoundError(
        f"{CFG['qa_set_path']} not found. Run Notebook 1 (Pure LLM) first to generate it."
    )

with open(CFG["qa_set_path"]) as f:
    qa_set = json.load(f)

print(f"QA set loaded: {len(qa_set)} questions")
for qa in qa_set[:5]:
    print(f"  [{qa['qa_id']:2d}] ({qa['topic']:8s}) {qa['question'][:65]}")

## 4. Embedding Model

`all-MiniLM-L6-v2` is 22 MB and runs entirely on CPU.  
It is the standard dense retrieval model for RAG benchmarks, making results comparable to published work.  
We load it once and reuse it for both corpus indexing and query embedding.

In [ ]:
print(f"Loading embedding model: {CFG['embed_model']}")
embedder = SentenceTransformer(CFG["embed_model"], device="cpu")
EMB_DIM  = embedder.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMB_DIM}")
print(f"Model size: ~22 MB on CPU")

## 5. Stream, Chunk, and Embed

**Why stream instead of loading to pandas:**  
Loading 120K articles to a DataFrame uses ~4 GB RAM. Streaming processes one article at a time — RAM stays flat.

**Chunking:** naive fixed-size character windows with overlap.  
We deliberately keep the chunker simple to isolate the effect of the **embedding and retrieval** quality, not the chunking sophistication.

Each chunk stores its `article_id` for traceability in the evaluation output.

In [ ]:
def chunk_text(text, size=CFG["chunk_size"], overlap=CFG["chunk_overlap"]):
    """Fixed-size character chunking with overlap. Returns list of strings."""
    chunks, i = [], 0
    while i < len(text):
        c = text[i : i + size].strip()
        if len(c) > 20:
            chunks.append(c)
        i += max(1, size - overlap)
    return chunks

# ── Stream dataset and build index in batches
print(f"Streaming {CFG['n_articles']} articles and building chunk index...")
stream = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train", streaming=True)

chunk_texts_list = []   # list of str
chunk_meta       = []   # list of {"article_id", "headline"}

article_count = 0
batch_texts   = []
batch_meta    = []

for row in tqdm(stream, total=CFG["n_articles"], desc="Chunking"):
    if article_count >= CFG["n_articles"]:
        break
    text = row.get("Article") or ""
    if not isinstance(text, str) or len(text.strip()) < 30:
        continue

    for c in chunk_text(text):
        batch_texts.append(c)
        batch_meta.append({"article_id": article_count,
                           "headline": str(row.get("Headline",""))})

    article_count += 1

    # Embed and flush every EMBED_BATCH articles worth of chunks
    if article_count % CFG["embed_batch"] == 0:
        chunk_texts_list.extend(batch_texts)
        chunk_meta.extend(batch_meta)
        batch_texts, batch_meta = [], []

# flush remainder
chunk_texts_list.extend(batch_texts)
chunk_meta.extend(batch_meta)

print(f"\nTotal chunks: {len(chunk_texts_list):,}")
print(f"Avg chunks / article: {len(chunk_texts_list) / max(article_count,1):.1f}")
print(f"Embedding {len(chunk_texts_list):,} chunks (this takes ~1-2 min)...")

BATCH = 512
all_vecs = []
for i in tqdm(range(0, len(chunk_texts_list), BATCH), desc="Embedding"):
    vecs = embedder.encode(
        chunk_texts_list[i:i+BATCH],
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=BATCH,
    )
    all_vecs.append(vecs)

index_matrix = np.vstack(all_vecs).astype(np.float32)  # (N_chunks, 384)
print(f"Index shape: {index_matrix.shape}")
print(f"Index RAM: ~{index_matrix.nbytes // 1024 // 1024} MB")

## 6. Retrieval Function

Cosine similarity via dot product on L2-normalised vectors — equivalent to FAISS `IndexFlatIP` but requires no extra install.  
`numpy` matrix multiply handles 15K × 384 in milliseconds.

In [ ]:
def retrieve(query: str, k: int = CFG["top_k"]) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.
    Returns list of {text, article_id, headline, score}.
    """
    q_vec = embedder.encode([query], normalize_embeddings=True)[0].astype(np.float32)
    sims  = index_matrix @ q_vec                       # (N_chunks,)
    top_i = np.argsort(-sims)[:k]
    return [
        {
            "text"      : chunk_texts_list[i],
            "article_id": chunk_meta[i]["article_id"],
            "headline"  : chunk_meta[i]["headline"],
            "score"     : float(sims[i]),
        }
        for i in top_i
    ]

# Sanity check
test = retrieve("Goldman Sachs acquisition deal")
print(f"Retrieval test — top {len(test)} chunks:")
for c in test:
    print(f"  [art {c['article_id']:4d}] score={c['score']:.3f}  {c['text'][:70]}...")

## 7. Vanilla RAG Inference

For each question:
1. Retrieve top-5 chunks by cosine similarity
2. Build a context block with source labels
3. Send to `mistral-nemo` with a grounding instruction

**Why the same LLM as the GraphRAG pipeline matters:**  
Any performance difference comes from the context quality (flat chunks vs graph subgraph), not from the model.

In [ ]:
RAG_PROMPT = """You are a financial analyst with access to retrieved Bloomberg news excerpts.
Answer the question using the provided context. Be specific and cite relevant facts.
If the context does not contain enough information, say so explicitly.

Retrieved context:
{context}

Question: {question}

Answer:"""

results = []
t0 = time.time()

print(f"Running Vanilla RAG on {len(qa_set)} questions (top-{CFG['top_k']} chunks)...")
print()

for qa in tqdm(qa_set, desc="Answering"):
    t_start  = time.perf_counter()
    chunks   = retrieve(qa["question"])
    ctx_lines = []
    for j, ch in enumerate(chunks, 1):
        ctx_lines.append(f"[Source {j} | Article {ch['article_id']} | sim={ch['score']:.3f}]\n{ch['text']}")
    context  = "\n\n".join(ctx_lines)

    answer   = ollama(CFG["llm_model"],
                      RAG_PROMPT.format(context=context, question=qa["question"]))
    latency  = (time.perf_counter() - t_start) * 1000

    # Did we retrieve the source article?
    source_hit = qa["article_id"] in [c["article_id"] for c in chunks]

    results.append({
        "qa_id"             : qa["qa_id"],
        "article_id"        : qa["article_id"],
        "topic"             : qa["topic"],
        "question"          : qa["question"],
        "reference_answer"  : qa["reference_answer"],
        "predicted_answer"  : answer,
        "context_used"      : [c["text"][:120] for c in chunks],
        "retrieved_art_ids" : [c["article_id"] for c in chunks],
        "source_hit"        : source_hit,
        "top1_score"        : round(chunks[0]["score"], 4) if chunks else 0.0,
        "latency_ms"        : round(latency, 1),
        "method"            : "vanilla_rag",
    })

elapsed = time.time() - t0
hit_rate = sum(r["source_hit"] for r in results) / len(results)
print(f"\nDone: {len(results)} answers in {elapsed:.0f}s ({elapsed/len(results):.1f}s each)")
print(f"Source article retrieved (top-{CFG['top_k']}): {hit_rate*100:.1f}% of questions")

## 8. RAG Triad Evaluation

Same prompts and same judge as Notebook 1 and the GraphRAG pipeline's `RAGTriadEvaluator`.  
For Vanilla RAG, Context Precision has meaning: it measures what fraction of the 5 retrieved chunks was actually useful.

In [ ]:
FAITH_PROMPT = (
    "Rate faithfulness of the answer to the context (0.0-1.0).\n"
    "Faithfulness = every claim in the answer is supported by the context.\n\n"
    "Context: {context}\nAnswer: {answer}\n\n"
    "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}"
)
RELEV_PROMPT = (
    "Rate relevance of the answer to the question (0.0-1.0).\n\n"
    "Question: {query}\nAnswer: {answer}\n\n"
    "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}"
)
PREC_PROMPT = (
    "Rate context precision (0.0-1.0).\n"
    "Precision = fraction of the context useful for answering the question.\n\n"
    "Question: {query}\nContext: {context}\n\n"
    "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}"
)

def parse_score(raw):
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return float(json.loads(m.group())["score"]) if m else 0.0
    except Exception:
        return 0.0

print(f"Evaluating {len(results)} answers with {CFG['judge_model']} judge...")
print()

for r in tqdm(results, desc="Triad eval"):
    ctx_preview = " | ".join(r["context_used"][:3])

    faith = parse_score(ollama(CFG["judge_model"],
        FAITH_PROMPT.format(context=ctx_preview, answer=r["predicted_answer"])))
    relev = parse_score(ollama(CFG["judge_model"],
        RELEV_PROMPT.format(query=r["question"], answer=r["predicted_answer"])))
    prec  = parse_score(ollama(CFG["judge_model"],
        PREC_PROMPT.format(query=r["question"], context=ctx_preview)))

    r["faithfulness"]      = round(faith, 3)
    r["answer_relevance"]  = round(relev, 3)
    r["context_precision"] = round(prec,  3)
    r["triad_avg"]         = round((faith + relev + prec) / 3, 3)

print()
print("=" * 60)
print("VANILLA RAG — BENCHMARK RESULTS")
print("=" * 60)
print(f"{'Metric':28s}  {'Mean':>7}  {'Std':>7}")
print("-" * 45)
for key in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
    vals = [r[key] for r in results]
    print(f"{key:28s}  {np.mean(vals):7.4f}  {np.std(vals):7.4f}")

hit_rate = float(np.mean([r["source_hit"] for r in results]))
print(f"{'source_hit_rate':28s}  {hit_rate:7.4f}")
print(f"{'latency_ms (avg)':28s}  {np.mean([r['latency_ms'] for r in results]):7.1f}")

## 9. Side-by-Side Preview vs Pure LLM

In [ ]:
llm_path = "baseline_llm_results.json"
if os.path.exists(llm_path):
    with open(llm_path) as f:
        llm_data = json.load(f)

    print("=" * 65)
    print("PRELIMINARY COMPARISON: Pure LLM vs Vanilla RAG")
    print("=" * 65)
    print(f"{'Metric':28s}  {'Pure LLM':>10}  {'Vanilla RAG':>12}  {'Delta':>8}")
    print("-" * 62)

    rag_means = {
        "faithfulness"     : float(np.mean([r["faithfulness"]      for r in results])),
        "answer_relevance" : float(np.mean([r["answer_relevance"]  for r in results])),
        "context_precision": float(np.mean([r["context_precision"] for r in results])),
        "triad_avg"        : float(np.mean([r["triad_avg"]         for r in results])),
    }

    for key in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
        v_llm = llm_data.get("mean_" + key.replace("answer_relevance","relevance")
                              .replace("context_precision","ctx_precision"), 0)
        # try exact key too
        v_llm = llm_data.get("mean_" + key, v_llm)
        v_rag = rag_means[key]
        delta = v_rag - v_llm
        sign  = "+" if delta >= 0 else ""
        print(f"{key:28s}  {v_llm:10.4f}  {v_rag:12.4f}  {sign}{delta:7.4f}")

    print(f"{'source_hit_rate (RAG only)':28s}  {'N/A':>10}  {hit_rate*100:11.1f}%")
    print()
    print("GraphRAG results will be added in the final comparison step.")
else:
    print("Pure LLM results not found — run Notebook 1 first.")

## 10. Save Results

In [ ]:
rag_means = {
    "faithfulness"     : float(np.mean([r["faithfulness"]      for r in results])),
    "answer_relevance" : float(np.mean([r["answer_relevance"]  for r in results])),
    "context_precision": float(np.mean([r["context_precision"] for r in results])),
    "triad_avg"        : float(np.mean([r["triad_avg"]         for r in results])),
    "source_hit_rate"  : float(np.mean([r["source_hit"]        for r in results])),
    "mean_latency_ms"  : float(np.mean([r["latency_ms"]        for r in results])),
}

summary = {
    "method"      : f"Vanilla RAG (mistral-nemo + all-MiniLM-L6-v2, top-{CFG['top_k']})",
    "n_questions" : len(results),
    "chunk_size"  : CFG["chunk_size"],
    "top_k"       : CFG["top_k"],
    **rag_means,
    "per_question": results,
}

with open(CFG["results_path"], "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {CFG['results_path']} ({os.path.getsize(CFG['results_path'])//1024} KB)")

## Summary

| What | Choice | Why |
|------|--------|-----|
| LLM | mistral-nemo via Ollama | Same as GraphRAG pipeline |
| Embedder | all-MiniLM-L6-v2 (22 MB, CPU) | Lightweight; standard RAG benchmark model |
| Vector store | numpy dot product | No FAISS install; handles 15K chunks in milliseconds |
| Chunking | 512 chars / 64 overlap | Preserves sentence coherence; same as platform backend |
| Top-k | 5 | Same as platform backend |
| Dataset | Bloomberg 120K, streamed, first 5 000 articles | Same corpus as pipeline, streamed to cap RAM |
| QA set | `qa_set.json` from Notebook 1 | Identical questions across all three systems |
| Judge | llama3.2:3b, temp=0 | Same as `RAGTriadEvaluator` in pipeline |

**Output files:**
- `baseline_rag_results.json` — per-question answers + all three triad scores

**Expected behaviour vs GraphRAG:**  
Vanilla RAG scores better than Pure LLM when the retriever finds the correct article (source_hit_rate).  
GraphRAG should score higher on **faithfulness** and **context precision** — its subgraph context is denser and more directly relevant than flat chunks, especially for multi-entity questions.